# 🚀 Quest 3S Stereo Super-Resolution: Epipolar Windowed DyT-NAFRSSR
### Dual NVIDIA Tesla T4 Accelerated Training Pipeline (32 GB VRAM)
- **Hardware Target:** Meta Quest 3S (Snapdragon XR2 Gen 2 / Adreno 740 / 1024 KB GMEM SRAM)
- **Architecture:** Epipolar Windowed DyT-NAFRSSR (82,368 parameters, 100% GMEM Tile Resident)
- **Dataset:** Flickr1024 Stereo Pairs (Train + Test splits)
- **Acceleration:** 2x Tesla T4 GPUs via `nn.DataParallel` with Automatic Mixed Precision (FP16)

## 1. System Environment & GPU Allocation Check

In [ ]:
!nvidia-smi
import torch
import os, sys, time, math, random, glob
from PIL import Image
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

num_gpus = torch.cuda.device_count()
print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] CUDA Available: {torch.cuda.is_available()} | Allocated GPUs: {num_gpus}")
for i in range(num_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f"    [GPU {i}] {p.name} | VRAM: {p.total_memory / (1024**3):.2f} GB | Compute: {p.major}.{p.minor}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 2. Dataset Loader with Fixed-Crop Collation (Crash-Proof)

In [ ]:
# Auto-detect attached dataset in /kaggle/input
DATASET_PATH = None
search_dirs = [
    '/kaggle/input/flickr1024/Flickr1024',
    '/kaggle/input/flickr1024',
    '/kaggle/input/datasets/tunt96/flickr1024/Flickr1024',
    '/kaggle/input/tunt96-flickr1024/Flickr1024'
]
for s in search_dirs:
    if os.path.exists(s):
        DATASET_PATH = s
        break

if not DATASET_PATH and os.path.exists('/kaggle/input'):
    for root, _, files in os.walk('/kaggle/input'):
        if any(f.endswith('_L.png') for f in files):
            DATASET_PATH = root
            break

print(f"[*] Mounted Dataset Root: {DATASET_PATH}")

class FixedCropStereoDataset(Dataset):
    """
    Loads stereo pairs and extracts fixed-size crops to guarantee identical tensor shapes in batches.
    Completely prevents 'stack expects each tensor to be equal size' RuntimeError.
    """
    def __init__(self, root_dir, crop_lr=48, upscale=4, is_train=True, max_pairs=800):
        self.crop_lr = crop_lr
        self.crop_hr = crop_lr * upscale
        self.upscale = upscale
        self.is_train = is_train
        self.pairs = []

        if root_dir and os.path.exists(root_dir):
            # Find all Left image files
            pattern = os.path.join(root_dir, '**', '*_L.png')
            left_files = sorted(glob.glob(pattern, recursive=True))
            for lf in left_files:
                rf = lf.replace('_L.png', '_R.png')
                if os.path.exists(rf):
                    self.pairs.append((lf, rf))
                    if len(self.pairs) >= max_pairs:
                        break

        print(f"[*] Loaded {len(self.pairs)} stereo pairs (is_train={is_train})")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        lf, rf = self.pairs[idx]
        img_l = Image.open(lf).convert('RGB')
        img_r = Image.open(rf).convert('RGB')

        arr_l = np.array(img_l, dtype=np.float32) / 255.0
        arr_r = np.array(img_r, dtype=np.float32) / 255.0

        t_l = torch.from_numpy(arr_l).permute(2, 0, 1)  # (3, H, W)
        t_r = torch.from_numpy(arr_r).permute(2, 0, 1)

        H, W = t_l.shape[1], t_l.shape[2]
        if self.is_train and H >= self.crop_hr and W >= self.crop_hr:
            top = random.randint(0, H - self.crop_hr)
            left = random.randint(0, W - self.crop_hr)
            p_hr_l = t_l[:, top:top+self.crop_hr, left:left+self.crop_hr]
            p_hr_r = t_r[:, top:top+self.crop_hr, left:left+self.crop_hr]
        else:
            # Central crop
            cy = max(0, (H - self.crop_hr) // 2)
            cx = max(0, (W - self.crop_hr) // 2)
            p_hr_l = t_l[:, cy:cy+self.crop_hr, cx:cx+self.crop_hr]
            p_hr_r = t_r[:, cy:cy+self.crop_hr, cx:cx+self.crop_hr]

        # Data augmentation for training
        if self.is_train:
            # Stereoscopic Horizontal Flip: Swap Left and Right eyes and flip horizontally
            if random.random() > 0.5:
                p_hr_l, p_hr_r = torch.flip(p_hr_r, dims=[2]), torch.flip(p_hr_l, dims=[2])
            # Vertical Flip
            if random.random() > 0.5:
                p_hr_l = torch.flip(p_hr_l, dims=[1])
                p_hr_r = torch.flip(p_hr_r, dims=[1])

        # Bicubic 4x downsample for LR inputs
        p_lr_l = F.interpolate(p_hr_l.unsqueeze(0), scale_factor=0.25, mode='bicubic', align_corners=False).squeeze(0).clamp(0, 1)
        p_lr_r = F.interpolate(p_hr_r.unsqueeze(0), scale_factor=0.25, mode='bicubic', align_corners=False).squeeze(0).clamp(0, 1)

        return {
            'lr_l': p_lr_l,
            'lr_r': p_lr_r,
            'hr_l': p_hr_l,
            'hr_r': p_hr_r
        }


## 3. Architecture Definition: Epipolar Windowed DyT-NAFRSSR
Features element-wise Dynamic Tanh (DyT) and 17-hypothesis Epipolar Cross-Attention covering $[-2, +14]$ disparity range.

In [ ]:
class DynamicTanh(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(1, channels, 1, 1))
        self.gamma = nn.Parameter(torch.ones(1, channels, 1, 1))
        self.beta = nn.Parameter(torch.zeros(1, channels, 1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.gamma * torch.tanh(self.alpha * x) + self.beta


class SimpleGate(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2


class WindowedSCAM(nn.Module):
    def __init__(self, channels: int, min_disp: int = -2, max_disp: int = 14):
        super().__init__()
        self.channels = channels
        self.min_disp = min_disp
        self.max_disp = max_disp
        self.scale = channels ** -0.5

        self.norm_l = DynamicTanh(channels)
        self.l_proj = nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1, groups=channels)
        self.beta = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.gamma = nn.Parameter(torch.zeros(1, channels, 1, 1))

    def forward(self, x_l: torch.Tensor, x_r: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        B, C, H, W = x_l.shape

        q_l = self.l_proj(self.norm_l(x_l))
        q_r = self.l_proj(self.norm_l(x_r))
        v_l = x_l
        v_r = x_r

        corr_l2r, shifted_v_r = [], []
        corr_r2l, shifted_v_l = [], []

        for offset in range(self.min_disp, self.max_disp + 1):
            if offset > 0:
                q_r_s = F.pad(q_r[:, :, :, :-offset], (offset, 0, 0, 0), mode='replicate')
                v_r_s = F.pad(v_r[:, :, :, :-offset], (offset, 0, 0, 0), mode='replicate')
                q_l_s = F.pad(q_l[:, :, :, offset:], (0, offset, 0, 0), mode='replicate')
                v_l_s = F.pad(v_l[:, :, :, offset:], (0, offset, 0, 0), mode='replicate')
            elif offset < 0:
                abs_off = abs(offset)
                q_r_s = F.pad(q_r[:, :, :, abs_off:], (0, abs_off, 0, 0), mode='replicate')
                v_r_s = F.pad(v_r[:, :, :, abs_off:], (0, abs_off, 0, 0), mode='replicate')
                q_l_s = F.pad(q_l[:, :, :, :-abs_off], (abs_off, 0, 0, 0), mode='replicate')
                v_l_s = F.pad(v_l[:, :, :, :-abs_off], (abs_off, 0, 0, 0), mode='replicate')
            else:
                q_r_s, v_r_s = q_r, v_r
                q_l_s, v_l_s = q_l, v_l

            sim_l2r = torch.sum(q_l * q_r_s, dim=1, keepdim=True) * self.scale
            corr_l2r.append(sim_l2r)
            shifted_v_r.append(v_r_s)

            sim_r2l = torch.sum(q_r * q_l_s, dim=1, keepdim=True) * self.scale
            corr_r2l.append(sim_r2l)
            shifted_v_l.append(v_l_s)

        weights_l2r = F.softmax(torch.cat(corr_l2r, dim=1), dim=1)
        f_r2l = torch.sum(torch.stack(shifted_v_r, dim=1) * weights_l2r.unsqueeze(2), dim=1)

        weights_r2l = F.softmax(torch.cat(corr_r2l, dim=1), dim=1)
        f_l2r = torch.sum(torch.stack(shifted_v_l, dim=1) * weights_r2l.unsqueeze(2), dim=1)

        out_l = x_l + f_r2l * self.beta
        out_r = x_r + f_l2r * self.gamma
        return out_l, out_r


class DyTNAFBlock(nn.Module):
    def __init__(self, c: int, dw_expand: int = 2, ffn_expand: int = 2):
        super().__init__()
        dw_channel = c * dw_expand
        self.conv1 = nn.Conv2d(c, dw_channel, kernel_size=1, bias=True)
        self.conv2 = nn.Conv2d(dw_channel, dw_channel, kernel_size=3, padding=1, groups=dw_channel, bias=True)
        self.conv3 = nn.Conv2d(dw_channel // 2, c, kernel_size=3, padding=1, groups=c // 4, bias=True)
        self.sg = SimpleGate()

        ffn_channel = ffn_expand * c
        self.conv4 = nn.Conv2d(c, ffn_channel, kernel_size=1, bias=True)
        self.conv5 = nn.Conv2d(ffn_channel // 2, c, kernel_size=1, bias=True)

        self.norm1 = DynamicTanh(c)
        self.norm2 = DynamicTanh(c)
        self.beta = nn.Parameter(torch.zeros(1, c, 1, 1))
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))

    def forward(self, inp: torch.Tensor) -> torch.Tensor:
        x = self.norm1(inp)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.sg(x)
        x = self.conv3(x)
        y = inp + x * self.beta

        x = self.norm2(y)
        x = self.conv4(x)
        x = self.sg(x)
        x = self.conv5(x)
        return y + x * self.gamma


class WindowedDyTNAFRSSR(nn.Module):
    def __init__(self, in_channels: int = 3, width: int = 48, num_blks: int = 4, up_scale: int = 4, min_disp: int = -2, max_disp: int = 14):
        super().__init__()
        self.up_scale = up_scale
        self.intro = nn.Conv2d(in_channels, width, kernel_size=3, padding=1, bias=True)
        self.blocks = nn.ModuleList([DyTNAFBlock(width) for _ in range(num_blks)])
        self.stereo_fusion = WindowedSCAM(width, min_disp=min_disp, max_disp=max_disp)
        self.up = nn.Sequential(
            nn.Conv2d(width, in_channels * (up_scale ** 2), kernel_size=3, padding=1, bias=True),
            nn.PixelShuffle(up_scale)
        )

    def forward(self, img_l: torch.Tensor, img_r: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        base_l = F.interpolate(img_l, scale_factor=self.up_scale, mode='bicubic', align_corners=False)
        base_r = F.interpolate(img_r, scale_factor=self.up_scale, mode='bicubic', align_corners=False)

        feat_l = self.intro(img_l)
        feat_r = self.intro(img_r)
        feat_l, feat_r = self.stereo_fusion(feat_l, feat_r)

        for blk in self.blocks:
            feat_l = blk(feat_l)
            feat_r = blk(feat_r)

        out_l = self.up(feat_l) + base_l
        out_r = self.up(feat_r) + base_r
        return out_l, out_r

print(f"[*] WindowedDyTNAFRSSR Architecture Defined! Total Params: {sum(p.numel() for p in WindowedDyTNAFRSSR().parameters()):,}")


## 4. Multi-GPU Distributed Training Loop (Dual Tesla T4)

In [ ]:
def calc_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    mse = F.mse_loss(pred.clamp(0, 1), target.clamp(0, 1)).item()
    if mse <= 1e-10:
        return 100.0
    return 10.0 * math.log10(1.0 / mse)

# Datasets
train_dataset = FixedCropStereoDataset(DATASET_PATH, crop_lr=48, upscale=4, is_train=True, max_pairs=800)
val_dataset = FixedCropStereoDataset(DATASET_PATH, crop_lr=48, upscale=4, is_train=False, max_pairs=50)

batch_size = 16 * max(1, num_gpus)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=1)

# Model & Parallelism
base_model = WindowedDyTNAFRSSR(width=48, num_blks=4, up_scale=4, min_disp=-2, max_disp=14).to(device)
if num_gpus > 1:
    print(f"[*] Wrapping model in nn.DataParallel across {num_gpus} Tesla T4 GPUs!")
    model = nn.DataParallel(base_model)
else:
    model = base_model

epochs = 20
optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
scaler = torch.amp.GradScaler('cuda')

best_val_psnr = 0.0
print(f"\n[*] Commencing 20-Epoch Training Run on {num_gpus}x Tesla T4 (Batch Size: {batch_size})...")
t_start = time.time()

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    t_ep = time.time()

    for step, batch in enumerate(train_loader):
        lr_l = batch['lr_l'].to(device, non_blocking=True)
        lr_r = batch['lr_r'].to(device, non_blocking=True)
        hr_l = batch['hr_l'].to(device, non_blocking=True)
        hr_r = batch['hr_r'].to(device, non_blocking=True)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            out_l, out_r = model(lr_l, lr_r)
            loss_l = torch.sqrt(torch.mean((out_l - hr_l)**2) + 1e-6)
            loss_r = torch.sqrt(torch.mean((out_r - hr_r)**2) + 1e-6)
            loss = loss_l + loss_r

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    dt = time.time() - t_ep

    # Validation
    model.eval()
    val_psnrs = []
    with torch.no_grad():
        for vbatch in val_loader:
            v_lr_l = vbatch['lr_l'].to(device)
            v_lr_r = vbatch['lr_r'].to(device)
            v_hr_l = vbatch['hr_l'].to(device)
            v_out_l, _ = model(v_lr_l, v_lr_r)
            val_psnrs.append(calc_psnr(v_out_l, v_hr_l))

    mean_val = float(np.mean(val_psnrs))
    if mean_val > best_val_psnr:
        best_val_psnr = mean_val
        torch.save(base_model.state_dict(), '/kaggle/working/dyt_nafrssr_quest3s_best.pth')

    print(f"[*] Epoch {epoch:02d}/{epochs:02d} ({dt:.1f}s) | Loss: {avg_loss:.5f} | Val PSNR: {mean_val:.2f} dB (Best: {best_val_psnr:.2f} dB)")

print(f"\n[OK] Training completed in {time.time() - t_start:.1f}s! Best Val PSNR: {best_val_psnr:.2f} dB")


## 5. Export Production ONNX (FP32 & FP16) for Quest 3S

In [ ]:
print("[*] Exporting Production ONNX Models...")
base_model.load_state_dict(torch.load('/kaggle/working/dyt_nafrssr_quest3s_best.pth', map_location=device))
base_model.eval()

dummy_l = torch.randn(1, 3, 64, 64, device=device)
dummy_r = torch.randn(1, 3, 64, 64, device=device)

# FP32 ONNX
onnx_fp32_path = '/kaggle/working/dyt_nafrssr_quest3s_fp32.onnx'
torch.onnx.export(
    base_model,
    (dummy_l, dummy_r),
    onnx_fp32_path,
    input_names=['left_eye_lr', 'right_eye_lr'],
    output_names=['left_eye_sr', 'right_eye_sr'],
    dynamic_axes={'left_eye_lr': {2: 'height', 3: 'width'}, 'right_eye_lr': {2: 'height', 3: 'width'}},
    opset_version=17
)
print(f"[OK] Exported FP32 ONNX: {onnx_fp32_path} ({os.path.getsize(onnx_fp32_path) / 1024:.1f} KB)")

# FP16 ONNX
onnx_fp16_path = '/kaggle/working/dyt_nafrssr_quest3s_fp16.onnx'
model_fp16 = WindowedDyTNAFRSSR(width=48, num_blks=4, up_scale=4, min_disp=-2, max_disp=14).to(device).half()
model_fp16.load_state_dict({k: v.half() for k, v in base_model.state_dict().items()})
dummy_l_16 = dummy_l.half()
dummy_r_16 = dummy_r.half()

torch.onnx.export(
    model_fp16,
    (dummy_l_16, dummy_r_16),
    onnx_fp16_path,
    input_names=['left_eye_lr', 'right_eye_lr'],
    output_names=['left_eye_sr', 'right_eye_sr'],
    dynamic_axes={'left_eye_lr': {2: 'height', 3: 'width'}, 'right_eye_lr': {2: 'height', 3: 'width'}},
    opset_version=17
)
print(f"[OK] Exported FP16 ONNX: {onnx_fp16_path} ({os.path.getsize(onnx_fp16_path) / 1024:.1f} KB)")

# Output directory verification
print("\n[*] Kaggle Output Directory Artifacts:")
for f in os.listdir('/kaggle/working'):
    sz = os.path.getsize(os.path.join('/kaggle/working', f)) / 1024
    print(f"    - {f:35s}: {sz:.1f} KB")
